# Power Flow: UI and API Usage
This tutorial shows how to:
1. Use the **web UI** to:
   - launch the power flow algorithm for a grid and phase
   - see the computed voltages (magnitude and angle) per node and time
2. Do the same **programmatically via the API** from Python.

Requirements:
- The backend is running, e.g.:
- You already have:
   - a grid registered (/grid router)
   - historical measurements for that grid (/historical router)

The power flow uses the grid topology + historical measurements as input, and writes results in "PowerFlowResults".

## 1. Using the Web UI
### 1.1. Open the Power Flow UI
1. Go to the home page:
   ```text
   http://localhost:8000/

2. Open the Power Flow page from the application UI.

![PF Menu](images/PF_menu.png)

On `/powerflow/ui` you should see:
- a **Run Power Flow** section
- an **Available Grids** table

### 1.2. Run Power Flow (UI form)
On `/powerflow/ui`, in the **Run Power Flow** section, you have:
- `Grid ID` (text input)
- `Phase` (select: R, S, T)
- `Start` (optional `datetime-local`)
- `End` (optional `datetime-local`)
- **Run** button

Fill the form:
1. **Grid ID** – must match an existing grid.
2. **Phase** – choose one of:
   - `R`
   - `S`
   - `T`
3. Optionally, set:
   - `Start`: only timestamps `>= start_time` are used
   - `End`: only timestamps `<= end_time` are used
4. Click **Run**.

![PF_Run](images/PF_run.png)

Under the hood, the browser sends a `POST` to:
```text
POST /powerflow/{grid_id}/run?phase=R&start_time=...&end_time=...

The backend:
1. Ensures the powerflow results table exists.
2. Queries `Measurements` for distinct timestamps for the selected `grid_id` and `phase`.
3. For each timestamp:
   - reads the node indexing
   - reads complex power injections
   - reads grid connections and admittances
   - reads the PT reference voltage
   - runs `full_pf(...)`
   - stores voltages in `PowerFlowResults`
    
When it finishes successfully, the endpoint returns:
```json
{"message": "Power flow has been run successfully."}

If no timestamps match the selected filters, the API raises an error with detail:

**`Error: No measurement timestamps found for the given range.`**

In the UI, that appears as an error message below the form.

### 1.3. Inspect Power Flow results per grid / phase
On `/powerflow/ui`, check the **Available Grids** card:
- It lists all `grid_id`s from the `grids` table.
- Each grid ID is a link to:
  ```text
  /powerflow/ui/{grid_id}

Click one, e.g. lv_grid_01:

http://localhost:8000/powerflow/ui/lv_grid_01

You’ll see a page with:
1. A header area showing the selected grid and a link back to `/powerflow/ui`
2. A **Phase Selection** section
   - select `R`, `S`, or `T`
   - changing the selection reloads the page for the same `grid_id`
3. A **Voltage Magnitude** Plotly chart
   - X-axis: datetime
   - Y-axis: voltage magnitude
   - one line per node
4. A **Voltage Angle** Plotly chart
   - X-axis: datetime
   - Y-axis: angle in degrees
   - one line per node
5. A **Data Table** section with columns:
   - `node_id`
   - `datetime`
   - `real`
   - `imag`
   - `magnitude`
   - `angle`

![PF_Sols](images/pf_solutions.png)

![PF_Sols2](images/pf_solutions_2.png)

If there are no stored results for that grid and phase, the page shows a **No Power Flow Results Found** message, a phase selector, and a button to go back to `/powerflow/ui`.

## 2. Using the API from Python
Let’s now call the same endpoints from a notebook using `requests` and `pandas`.

In [ ]:
import requests
import pandas as pd
from datetime import datetime

BASE_URL = "http://localhost:8000"  # adjust if needed


def check_response(resp: requests.Response):
    """Helper to raise errors and return parsed JSON or text."""
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        try:
            print("Error payload:", resp.json())
        except Exception:
            print("Raw response:", resp.text)
        raise e
    try:
        return resp.json()
    except Exception:
        return resp.text

### 2.1. Run Power Flow programmatically (`POST /powerflow/{grid_id}/run`)
Endpoint:
```text
POST /powerflow/{grid_id}/run?phase=R&start_time=...&end_time=...

Parameters:
- grid_id (path): which grid to analyse
- phase (query, required): "R", "S", "T"
- start_time (query, optional): ISO 8601 string for lower bound
- end_time (query, optional): ISO 8601 string for upper bound

The endpoint:
- finds all measurement timestamps for grid_id and phase in "Measurements" within the window.
- runs single_timestamp_power_flow(...) for each timestamp.
- writes results into "PowerFlowResults".
- returns a message on success.

We’ll wrap this in a helper:

In [ ]:
def run_power_flow(
    grid_id: str,
    phase: str,
    start_time: datetime | None = None,
    end_time: datetime | None = None,
):
    """
    Call POST /powerflow/{grid_id}/run to compute power flow for a time window.
    """
    url = f"{BASE_URL}/powerflow/{grid_id}/run"
    params: dict[str, str] = {"phase": phase}
    if start_time:
        params["start_time"] = start_time.isoformat()
    if end_time:
        params["end_time"] = end_time.isoformat()
    resp = requests.post(url, params=params)
    return check_response(resp)
# Example (uncomment and adapt):
# run_power_flow(
#     "lv_grid_01",
#     phase="R",
#     start_time=datetime(2025, 7, 31, 0, 0, 0),
#     end_time=datetime(2025, 7, 31, 23, 59, 59),
# )

Error situations:
- If there is no historical data for that grid/phase window, the endpoint raises and you get a 500 with detail "Error: No measurement timestamps found for the given range."
- Any internal DB or power flow error also returns status 500.

### 2.2. Retrieve stored results (`GET /powerflow/data/{grid_id}/results`)

Endpoint:
```text
GET /powerflow/data/{grid_id}/results?phase=R

Query parameter:
- phase: "R", "S", or "T" (required)

It returns:
```json
{
  "grid_id": "lv_grid_01",
  "phase": "R",
  "results": [
    {
      "timestamp": "...",
      "node_id": "N001",
      "voltage": {
        "real": 230.0,
        "imag": -1.2
      }
    },
    ...
  ]
}

We’ll request this and convert to a pandas DataFrame with magnitude/angle.

In [ ]:
import numpy as np


def get_power_flow_results(grid_id: str, phase: str) -> pd.DataFrame:
    """
    Call GET /powerflow/data/{grid_id}/results and return a DataFrame with:
    node_id, datetime, real, imag, magnitude, angle_deg
    """
    url = f"{BASE_URL}/powerflow/data/{grid_id}/results"
    params = {"phase": phase}
    data = check_response(requests.get(url, params=params))
    results = data.get("results", [])
    if not results:
        return pd.DataFrame()

    flattened = [
        {
            "node_id": r["node_id"],
            "datetime": r["timestamp"],
            "real": r["voltage"]["real"],
            "imag": r["voltage"]["imag"],
        }
        for r in results
    ]
    df = pd.DataFrame(flattened)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["magnitude"] = np.sqrt(df["real"] ** 2 + df["imag"] ** 2)
    df["angle_deg"] = np.degrees(np.arctan2(df["imag"], df["real"]))
    return df

# Example (after running power flow):
# df_pf = get_power_flow_results("lv_grid_01", "R")
# df_pf.head()

If no rows exist for that grid + phase, you get:
- HTTP 404 with detail "No power flow results found."

### 2.3. Quick plots in Python
The UI already gives you Plotly plots, but you can also plot in the notebook.

Example: voltage magnitude over time for a single node:

In [ ]:
import matplotlib.pyplot as plt
def plot_magnitude_for_node(grid_id: str, phase: str, node_id: str):
    df = get_power_flow_results(grid_id, phase)
    if df.empty:
        print("No power flow results available.")
        return
    node_df = df[df["node_id"] == node_id]
    if node_df.empty:
        print(f"No results for node {node_id} in phase {phase}.")
        return
    plt.figure()
    plt.plot(node_df["datetime"], node_df["magnitude"])
    plt.xlabel("Datetime")
    plt.ylabel("Voltage magnitude (V)")
    plt.title(f"{grid_id} – Node {node_id} – Phase {phase}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
# Example:
# plot_magnitude_for_node("lv_grid_01", "R", "N001")

Or, magnitude for all nodes in a single timestamp (e.g. snapshot):

In [ ]:
def plot_snapshot(grid_id: str, phase: str, timestamp: datetime):
    df = get_power_flow_results(grid_id, phase)
    if df.empty:
        print("No power flow results available.")
        return
    snap = df[df["datetime"] == timestamp]
    if snap.empty:
        print(f"No results at timestamp {timestamp}.")
        return
    snap_sorted = snap.sort_values("node_id")
    plt.figure()
    plt.stem(
        range(len(snap_sorted)),
        snap_sorted["magnitude"],
        use_line_collection=True,
    )
    plt.xticks(range(len(snap_sorted)), snap_sorted["node_id"], rotation=90)
    plt.ylabel("Voltage magnitude (V)")
    plt.title(f"{grid_id} – Phase {phase} – {timestamp}")
    plt.tight_layout()
    plt.show()
# Example:
# ts = df_pf["datetime"].min()
# plot_snapshot("lv_grid_01", "R", ts)

## 3. End-to-end workflow

Putting everything together:

1. **Prepare inputs**
   - Use `/grid` to register the grid.
   - Use `/historical` to upload historical measurements for that grid.

2. **Run power flow (UI or API)**

   **UI**
   - Go to `/powerflow/ui`
   - Fill `Grid ID`, `Phase`, and optionally `Start` / `End`
   - Click **Run**

   **Python**
   ```python
   run_power_flow(
       "lv_grid_01",
       phase="R",
       start_time=datetime(2025, 7, 31),
       end_time=datetime(2025, 8, 1),
   )

The call returns a success message when the computation and persistence complete.

3. **Inspect results**

    UI:
    - Go to /powerflow/ui
    - Click the grid ID
    - Use the phase selector to switch between R, S, and T
    - Inspect the charts and table

   ```python
      df_pf = get_power_flow_results("lv_grid_01", "R")
      df_pf.head()

4. **Analyse / post-process**
    - Use pandas and matplotlib in the notebook, or
    - export df_pf to CSV or Parquet for downstream tools

---